# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [12]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：data\E Commerce Dataset.xlsx
项目输出目录：C:\Users\和桢楠\PyCharmMiscProject\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [13]:
# 在此写下你的答案：
# 1.每条记录代表一位电商用户的个人档案与行为样本，单行为单个独立用户的全量信息。
# 2.该用户流失预测项目中，目标变量一般为Churn（用户是否流失）列，是模型最终需要预测的标签。
# 3.CustomerID是用户唯一身份编号，属于分类标识型ID字段，并非具备数值大小意义的连续变量：
# 数字大小不代表用户特征强弱，仅用于区分个体；
# 若当作连续数值计算均值、方差、回归系数，完全无业务意义，还会干扰模型训练；
# 该字段仅用于数据匹配、去重、样本追溯，不能参与特征建模运算

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [14]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    dtype_series = data.dtypes
    missing_count = data.isnull().sum()

    missing_rate = round((data.isnull().mean() * 100), 2)
    unique_counts = data.nunique()
    report_df = pd.DataFrame({
        "数据类型": dtype_series,
        "缺失数量":missing_count,
        "缺失比例":missing_rate,
        "唯一值数量":unique_counts
    })
    return report_df


# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例,唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [15]:
# TODO：完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：",raw_df["CustomerID"].duplicated().sum())
print(raw_df["Churn"].value_counts())
print("流失率：", round(raw_df["Churn"].mean() * 100, 2), "%")
#
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
     print(f"\n{col}")
     print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.84 %

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [16]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [17]:
def clean_ecommerce_data(data):
    import pandas as pd

def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # 1. 复制数据，避免覆盖原始数据
    df = data.copy()
    logs = []
    step_num = 1

    # 2. 步骤1：删除完全重复行
    before_cnt = len(df)
    df = df.drop_duplicates()
    after_cnt = len(df)
    affect_cnt = before_cnt - after_cnt
    logs.append({
        "处理步骤": step_num,
        "处理规则": "删除整行完全重复记录",
        "处理前记录数": before_cnt,
        "处理后记录数": after_cnt,
        "影响记录数": affect_cnt
    })
    step_num += 1

    # 3. 步骤2：数值型缺失值用中位数填充（示例常量，可根据实际列替换）
    NUMERIC_MISSING_COLS = ["Tenure", "WarehouseToHome", "HourSpendOnApp", "NumberOfDeviceRegistered",
                            "SatisfactionScore", "NumberOfAddress", "OrderAmountHikeFromlastYear",
                            "CouponUsed", "OrderCount", "DaySinceLastOrder", "CashbackAmount"]
    for col in NUMERIC_MISSING_COLS:
        if col in df.columns:
            before_col_cnt = len(df)
            missing_num = df[col].isna().sum()
            df[col] = df[col].fillna(df[col].median())
            logs.append({
                "处理步骤": step_num,
                "处理规则": f"字段 {col} 缺失值采用中位数填充",
                "处理前记录数": before_col_cnt,
                "处理后记录数": before_col_cnt,
                "影响记录数": missing_num
            })
    step_num += 1

    # 4. 步骤3：类别字段标准化（统一大小写、合并同义文本）
    cat_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat", "MaritalStatus"]
    for col in cat_cols:
        if col in df.columns:
            before_cnt = len(df)
            # 统一小写去空格
            df[col] = df[col].astype(str).str.strip().str.lower()
            affect_cnt = (df[col] != data.copy()[col].astype(str).str.strip().str.lower()).sum()
            logs.append({
                "处理步骤": step_num,
                "处理规则": f"类别字段 {col} 标准化：去除首尾空格、统一小写",
                "处理前记录数": before_cnt,
                "处理后记录数": before_cnt,
                "影响记录数": affect_cnt
            })
    step_num += 1

    # 5. 步骤4：必要数据类型转换
    # 示例：CustomerID、Churn转为整型
    type_cols = ["CustomerID", "Churn"]
    for col in type_cols:
        if col in df.columns:
            before_cnt = len(df)
            df[col] = df[col].astype(int)
            logs.append({
                "处理步骤": step_num,
                "处理规则": f"字段 {col} 转换为int数值类型",
                "处理前记录数": before_cnt,
                "处理后记录数": before_cnt,
                "影响记录数": before_cnt
            })
    step_num += 1

    # 日志转为DataFrame
    cleaning_log = pd.DataFrame(logs)
    cleaned_df = df

    return cleaned_df, cleaning_log
    """"
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    # TODO：创建日志列表 logs
    # TODO：删除完全重复行，并记录日志
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    # TODO：将 Churn 和 Complain 转为整数类型
    # TODO：返回 cleaned_df 与 cleaning_log

### 任务 3：运行清洗函数并查看日志

In [18]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,1,删除整行完全重复记录,5630,5630,0
1,2,字段 Tenure 缺失值采用中位数填充,5630,5630,264
2,2,字段 WarehouseToHome 缺失值采用中位数填充,5630,5630,251
3,2,字段 HourSpendOnApp 缺失值采用中位数填充,5630,5630,255
4,2,字段 NumberOfDeviceRegistered 缺失值采用中位数填充,5630,5630,0
5,2,字段 SatisfactionScore 缺失值采用中位数填充,5630,5630,0
6,2,字段 NumberOfAddress 缺失值采用中位数填充,5630,5630,0
7,2,字段 OrderAmountHikeFromlastYear 缺失值采用中位数填充,5630,5630,265
8,2,字段 CouponUsed 缺失值采用中位数填充,5630,5630,256
9,2,字段 OrderCount 缺失值采用中位数填充,5630,5630,258


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,mobile phone,3,6.00,debit card,Female,3.00,3,laptop & accessory,2,single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,phone,1,8.00,upi,Male,3.00,4,mobile,3,single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,phone,1,30.00,debit card,Male,2.00,4,mobile,3,single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,phone,3,15.00,debit card,Male,2.00,4,laptop & accessory,5,single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,phone,1,12.00,cc,Male,3.00,3,mobile,5,single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [19]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }
df = cleaned_df.copy()

# 1. 构造时长分层，新增 TenureGroup
tenure_bins = [0, 6, 12, float("inf")]
tenure_labels = ["短期(0-6)", "中期(7-12)", "长期(12+)"]
df["TenureGroup"] = pd.cut(df["Tenure"], bins=tenure_bins, labels=tenure_labels)

# 2. 新增 IsMobileLogin：移动端为1，其余0
mobile_words = ["mobile phone", "phone"]
df["IsMobileLogin"] = df["PreferredLoginDevice"].isin(mobile_words).astype(int)

# 3. 生成异常值报告：WarehouseToHome、OrderCount、CashbackAmount
check_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_result = []
for col in check_cols:
    info = iqr_outlier_summary(df[col])
    info["字段名"] = col
    outlier_result.append(info)

outlier_report = pd.DataFrame(outlier_result)
outlier_report = outlier_report[["字段名", "Q1", "Q3", "下限", "上限", "候选异常值数量"]]
display(outlier_report)

# 赋值为 transformed_df，供后续业务规则检查使用
transformed_df = df
cleaned_df["TenureGroup"] = df["TenureGroup"]
cleaned_df["IsMobileLogin"] = df["IsMobileLogin"]
# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
# TODO：生成 outlier_report（每行对应一个待检查字段）

,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [20]:
# TODO：完成业务规则检查
import pandas as pd
from IPython.display import display
transformed_df = cleaned_df.copy()

# 计算各规则不合规数量
rule1_count = (transformed_df["Tenure"] < 0).sum()                # 使用时长小于0
rule2_count = (transformed_df["WarehouseToHome"] < 0).sum()       # 仓库距离小于0
rule3_count = (transformed_df["OrderCount"] <= 0).sum()           # 订单数小于或等于0
rule4_count = (transformed_df["CashbackAmount"] < 0).sum()       # 返现金额小于0

# 构造业务规则检查报表
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长Tenure小于0",
        "仓库距离WarehouseToHome小于0",
        "订单数量OrderCount小于或等于0",
        "返现金额CashbackAmount小于0"
    ],
    "不合规记录数": [rule1_count, rule2_count, rule3_count, rule4_count]
})
display(business_rule_report)
# 处理结论：1. 以上四项指标均属于物理业务上不可能为负数的字段，负值属于录入错误、脏数据；
#2. 若存在不合规记录：该类数据无业务解释意义，建议做剔除处理，避免干扰统计建模；
#3. 若不合规记录全部为0：说明当前数据集在四项基础业务逻辑上完全合规，无需额外清洗操作，该结果同样记入项目日志留存备查。

,规则,不合规记录数
0,使用时长Tenure小于0,0
1,仓库距离WarehouseToHome小于0,0
2,订单数量OrderCount小于或等于0,0
3,返现金额CashbackAmount小于0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [21]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)

assert transformed_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in transformed_df["PreferredLoginDevice"].unique()
assert "COD" not in transformed_df["PreferredPaymentMode"].unique()
assert "CC" not in transformed_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(transformed_df.columns)

# TODO：导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")
print("===== 异常值报告 outlier_report =====")
display(outlier_report)

print("\n===== 业务规则检查报告 business_rule_report =====")
display(business_rule_report)

# TODO：打印全部交付文件路径
print("\n===== 全部交付文件路径 =====")
file_list = [
    OUTPUT_DIR / "data_quality_before.csv",
    OUTPUT_DIR / "data_quality_after.csv",
    OUTPUT_DIR / "cleaning_log.csv",
    OUTPUT_DIR / "ecommerce_customer_cleaned.csv"
]
for path in file_list:
    print(path.resolve())
# TODO：输出 outlier_report 和 business_rule_report
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")
# TODO：输出交付文件的路径

===== 异常值报告 outlier_report =====


,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438



===== 业务规则检查报告 business_rule_report =====


,规则,不合规记录数
0,使用时长Tenure小于0,0
1,仓库距离WarehouseToHome小于0,0
2,订单数量OrderCount小于或等于0,0
3,返现金额CashbackAmount小于0,0



===== 全部交付文件路径 =====
C:\Users\和桢楠\PyCharmMiscProject\output\day04_project\data_quality_before.csv
C:\Users\和桢楠\PyCharmMiscProject\output\day04_project\data_quality_after.csv
C:\Users\和桢楠\PyCharmMiscProject\output\day04_project\cleaning_log.csv
C:\Users\和桢楠\PyCharmMiscProject\output\day04_project\ecommerce_customer_cleaned.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

1.发现的问题：存在字段缺失值、同一类别表述不统一、数值字段出现极端候选异常值、格式错乱等数据质量缺陷。

2.处理策略：缺失值按字段重要性分删除/填充；类别不一致做统一标准化映射；候选异常值结合业务做截断/剔除并留存清洗日志。

3.清洗后数据缺失、格式、异常问题得到规范修正，留存完整清洗日志可溯源，满足后续分析的数据规范性要求。

4.极端阈值划定、缺失字段填充方式、小众类别合并规则，需要业务人员结合行业实际进一步确认。